# Preprocessing

This notebook will be used to explore preprocessing steps for the lfw and the wider faces datasets to be used to train the yolov8 and siamese network

In [36]:
# general functions
import matplotlib.pyplot as plt
from pathlib import Path
import shutil
import yaml
import os
import pandas as pd
import numpy as np
from PIL import Image

# YOLOv8 dependencies
from ultralytics import YOLO
import cv2

## YoloV8 formatting

YOLOv8 Format asks the directory to be in this kind of format:  
class_id | x_center | y_center | width | height  
with class_id being alawys 0, since there are only pictures with faces in wider faeces. Width and height are both normalized. 

While the format of the wider faces annotations for each image is in this general format:  
image path  
number of faces  
x1, y1, w, h, blur, expression, illumination, invalid, occlusion, pose

The wider faces dataset was originally used for a contest, so we're going to discard any portion of the dataset that doesn't have annotations, which is the testing files the example submission. Also going to discard the MATLAB format files from the split file. This leaves us with just 4 files that has the correct bounding boxes:  
- Wider_train  
- Wider_val  
- wider_face_val_bbx_gt.txt
- wider_face_train_bbx_gt.txt

Yolov8 formatting also wants a YAML file  

In [14]:
def yolo_dirs(target="/data/", type="train", remove=False):
    """
    removes and then creates yolo_dirs and file structure

    Args:
        target (str): the target directory
        names (str): what type of file is under that, ie. training/val/test 
    
    returns: names of the image and labels path
    """
    root = os.path.join(target, "yolov8_dataset")
    # remove if remove is set to true
    if os.path.exists(root) and remove:
        shutil.rmtree(root)
        print("directory removed")
        return
    
    # make the target path of the yolov8 dataset
    if os.path.exists(root):
        print(f"Directory {root} already exists")
    else:
        os.makedirs(root)

    labels_path = os.path.join(root, "labels")
    imgs_path = os.path.join(root, "images")
    # labels and images folders
    if os.path.exists(labels_path):
        print(f"Directory {labels_path} already exists")
    else:
        os.makedirs(labels_path)

    if os.path.exists(imgs_path):
        print(f"Directory {imgs_path} already exists")
    else:
        os.makedirs(imgs_path)
    
    labels_type_path = os.path.join(labels_path,type)
    imgs_type_path = os.path.join(imgs_path,type)
    # now make the type of img and label
    if os.path.exists(labels_type_path):
        print(f"Directory {labels_type_path} already exists")
    else:
        os.makedirs(labels_type_path)

    if os.path.exists(imgs_type_path):
        print(f"Directory {imgs_type_path} already exists")
    else:
        os.makedirs(imgs_type_path)

    return root, labels_type_path, imgs_type_path

In [23]:
# function testing
target = "../data/"
yolo_dirs(target,remove=True)
yolo_dirs(target, "train")
yolo_dirs(target, "val")
yolo_dirs(target, "val")

Directory ../data/yolov8_dataset already exists
Directory ../data/yolov8_dataset/labels already exists
Directory ../data/yolov8_dataset/images already exists
Directory ../data/yolov8_dataset/labels/train already exists
Directory ../data/yolov8_dataset/images/train already exists
Directory ../data/yolov8_dataset already exists
Directory ../data/yolov8_dataset/labels already exists
Directory ../data/yolov8_dataset/images already exists
Directory ../data/yolov8_dataset already exists
Directory ../data/yolov8_dataset/labels already exists
Directory ../data/yolov8_dataset/images already exists
Directory ../data/yolov8_dataset/labels/val already exists
Directory ../data/yolov8_dataset/images/val already exists


('../data/yolov8_dataset',
 '../data/yolov8_dataset/labels/val',
 '../data/yolov8_dataset/images/val')

In [53]:
def create_yaml(path, test_path=None, classes=["face", "dick", "balls"]):
    """
    Creates a yaml file based on the path, assumes that there are the required files (train and val)
    supports optional test images

    Args:
        path (str): the path to the yolov8 dataset
    """
    if os.path.exists(os.path.join(path,"data.yaml")):
        os.remove(os.path.join(path,"data.yaml"))
        print("Removed previous yaml file")
    
    # class names to dict
    class_dict = []
    for i in range(len(classes)):
        class_dict.append({i:classes[i]})

    # the dict that will be dumped into the yaml file later
    data = {}
    data = {
        "path":path,
        "train":"images/train",
        "val":"images/val",
    }
    if test_path:
        data["test"] = test_path
    data["names"] = class_dict
    
    # take a massive dump
    with open(os.path.join(path,"data.yaml"),'w') as file:
        yaml.dump(data, file)
    
    print("yaml file created")

In [54]:
create_yaml('../data/yolov8_dataset')

yaml file created


In [18]:
def create_yolo_annotation(chunk, img_width, img_height):
    """
    returns a list of the center_x center_y width height that follows yolo formatting

    Args:
        chunk (list): list of list of integers. It is the x1, y1, w, h parts from the wider faces annotation file

    """
    res = []
    for i in chunk:
        i = [int(j) for j in i]
        x1, y1, w, h = i
        center_x = (x1+w/2) / img_width
        center_y = (y1+h/2) / img_height
        width = w / img_width
        height = h / img_height
        # class id is always 0 since there is only one class, which is a face
        res.append([0, center_x, center_y, width, height])
    return res 

In [19]:
# transforms the wider faces into yolov8 format
def populate_yolo(root_boxes, root_images, target_img, target_labels):
    """
    takes the annotation paths and the images path, as well as the target images folder and the target labels folder, changes the format from widerfaces to yolov8 and then
    transfers the images (if possible) to the target_imgs folder

    Args:
        root_boxes (str): 

    """
    with open(root_boxes, "r") as file:
        annotations = file.readlines()